# Scraping Intro Homework: Wikipedia Table

In this assignment, we'll be extracting data from Wikipedia's table of the tallest buildings in Brooklyn: https://en.wikipedia.org/wiki/List_of_tallest_buildings_in_Brooklyn

### 0) Setup

Import `requests`, `BeautifulSoup`, and `pandas`. Although this homework uses `BeautifulSoup`, you can choose to use `lxml` instead, if you prefer.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

### 1) Grab the HTML for the webpage linked above

Use `requests` to get the HTML, assigning it to a variable

In [4]:
url = requests.get('https://en.wikipedia.org/wiki/List_of_tallest_buildings_in_Brooklyn')
html = url.text

### 2) Convert the HTML into a `BeautifulSoup` object

In [6]:
soup = BeautifulSoup(html)
soup

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-sticky-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-enabled vector-feature-main-menu-pinned-disabled vector-feature-limited-width-enabled vector-feature-limited-width-content-enabled vector-feature-zebra-design-disabled" dir="ltr" lang="en">
<head>
<meta charset="utf-8"/>
<title>List of tallest buildings in Brooklyn - Wikipedia</title>
<script>document.documentElement.className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-sticky-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-enabled vector-feature-main-menu-pinned-disabled vector-feature-limited-width-enabled vector-feature-limited-width-content-enabled vector-feature-zebra-design-disabled";(function(){var cookie=document.cookie.match(/(?

### 3) Use `.select(...)` (and `[0]`) to select the main table

That's the one directly under the "Tallest buildings" heading.

Print out the first 100 characters of text from the table to make sure you have the right one.

In [20]:
table = soup.select('table tr')
table

[<tr>
 <th>Rank
 </th>
 <th>Name<sup class="reference" id="ref_id1none"><a href="#endnote_id1none">[a]</a></sup>
 </th>
 <th class="unsortable">Image
 </th>
 <th data-sort-type="number">Height<br/><span style="font-size:85%;">ft (m)</span>
 </th>
 <th>Floors
 </th>
 <th>Year completed
 </th>
 <th class="unsortable">Notes
 </th></tr>,
 <tr>
 <td style="word-spacing: -5px;">1
 </td>
 <td><a href="/wiki/The_Brooklyn_Tower" title="The Brooklyn Tower">The Brooklyn Tower</a>
 </td>
 <td><a class="image" href="/wiki/File:The_Brooklyn_Tower_010.jpg" title="City Point"><img alt="A view of The Brooklyn Tower looking north from Bond Street" data-file-height="2048" data-file-width="1536" decoding="async" height="107" src="//upload.wikimedia.org/wikipedia/commons/thumb/3/36/The_Brooklyn_Tower_010.jpg/80px-The_Brooklyn_Tower_010.jpg" srcset="//upload.wikimedia.org/wikipedia/commons/thumb/3/36/The_Brooklyn_Tower_010.jpg/120px-The_Brooklyn_Tower_010.jpg 1.5x, //upload.wikimedia.org/wikipedia/commons/t

### 4) Use `.select(...)` (and `[0]`) again to select the table's first row

... which is its header. (Reminder that the `<thead>` tag is optional. Wikipedia tables don't use it.)

In [59]:
soup.select('table tr')[0]('th')

[<th>Rank
 </th>,
 <th>Name<sup class="reference" id="ref_id1none"><a href="#endnote_id1none">[a]</a></sup>
 </th>,
 <th class="unsortable">Image
 </th>,
 <th data-sort-type="number">Height<br/><span style="font-size:85%;">ft (m)</span>
 </th>,
 <th>Floors
 </th>,
 <th>Year completed
 </th>,
 <th class="unsortable">Notes
 </th>]

### 5) Extract the column names from that header

Use `.strip()` to remove any leading or trailing whitespace from the names.

First, try doing this with a standard `for` loop:

In [68]:
headers = []

for header in soup.select('table tr')[0]('th'):
    headers.append(header.text.strip())

headers

['Rank',
 'Name[a]',
 'Image',
 'Heightft (m)',
 'Floors',
 'Year completed',
 'Notes']

Now try to do it with a list comprehension:

In [113]:
headers = [header.text.strip() for header in soup.select('table tr')[0]('th')]
headers

['Rank',
 'Name[a]',
 'Image',
 'Heightft (m)',
 'Floors',
 'Year completed',
 'Notes']

### 6) Select all non-header row *elements* from the table

Since the header was the first row, you'll want to skip that one. How many rows are there? (Check with your eyes that this number matches what you deduce from the rankings column in the browser-rendered table.)

In [78]:
len(soup.select('table')[0]('tr')[1:])

73

### 7) For each row, extract the text of each cell into a Python list

First, try this as two nested `for` loops:

In [118]:
buildings = []
rows = []

for row in soup.select('table')[0]('tr')[1:]:
    for cell in row.select('td'):
        rows.append(cell.text.strip())
    buildings.append(rows)

buildings

[['1',
  'The Brooklyn Tower',
  '',
  '1,073 (327)',
  '93',
  '2022',
  'Topped out in October 2021.[2][22][23][24]',
  '2',
  'Brooklyn Point',
  '',
  '720 (219)',
  '68',
  '2019',
  "The final phase of Extell's City Point development; topped out in April 2019, it is now the second tallest building in Brooklyn.[25] Also known as 138 Willoughby Street,[26][27] 1 City Point,[28] and City Point Tower III.[28][29][30]",
  '3',
  '11 Hoyt',
  '',
  '626 (191)',
  '51',
  '2020',
  "Topped out in June 2019.[31] A redevelopment of Macy's former footprint in Downtown Brooklyn, with a design seemingly inspired by 8 Spruce Street.[32][33]",
  '4',
  'The Hub',
  '',
  '611 (186)',
  '52',
  '2017',
  'Also known as 333 Schermerhorn Street. Topped out on December 16, 2015.[34][35][36][37][38]',
  '5',
  'AVA DoBro',
  '',
  '596 (182)',
  '58',
  '2015',
  'Also known as 100 Willoughby Street, Avalon Willoughby Square, and 214 Duffield Street.[39][40][41]',
  '6',
  '388 Bridge Street',
  ''

Now try with two list comprehensions, one nested in the other:

In [128]:
buildings = [[cell.text.strip() for cell in row.select('td')] for row in soup.select('table')[0]('tr')[1:]]

# buildings = [[cell.text.strip() for cell in row.select('td')] for row in soup.select('table')0[1:]

buildings

[['1',
  'The Brooklyn Tower',
  '',
  '1,073 (327)',
  '93',
  '2022',
  'Topped out in October 2021.[2][22][23][24]'],
 ['2',
  'Brooklyn Point',
  '',
  '720 (219)',
  '68',
  '2019',
  "The final phase of Extell's City Point development; topped out in April 2019, it is now the second tallest building in Brooklyn.[25] Also known as 138 Willoughby Street,[26][27] 1 City Point,[28] and City Point Tower III.[28][29][30]"],
 ['3',
  '11 Hoyt',
  '',
  '626 (191)',
  '51',
  '2020',
  "Topped out in June 2019.[31] A redevelopment of Macy's former footprint in Downtown Brooklyn, with a design seemingly inspired by 8 Spruce Street.[32][33]"],
 ['4',
  'The Hub',
  '',
  '611 (186)',
  '52',
  '2017',
  'Also known as 333 Schermerhorn Street. Topped out on December 16, 2015.[34][35][36][37][38]'],
 ['5',
  'AVA DoBro',
  '',
  '596 (182)',
  '58',
  '2015',
  'Also known as 100 Willoughby Street, Avalon Willoughby Square, and 214 Duffield Street.[39][40][41]'],
 ['6', '388 Bridge Street', '

### 8) Turn the data you've extracted into a `pandas` `DataFrame`

In [257]:
df = pd.DataFrame(buildings, columns=headers)
df

,Rank,Name[a],Image,Heightft (m),Floors,Year completed,Notes
0,1,The Brooklyn Tower,,"1,073 (327)",93,2022,Topped out in October 2021.[2][22][23][24]
1,2,Brooklyn Point,,720 (219),68,2019,The final phase of Extell's City Point develop...
2,3,11 Hoyt,,626 (191),51,2020,Topped out in June 2019.[31] A redevelopment o...
3,4,The Hub,,611 (186),52,2017,Also known as 333 Schermerhorn Street. Topped ...
4,5,AVA DoBro,,596 (182),58,2015,"Also known as 100 Willoughby Street, Avalon Wi..."
...,...,...,...,...,...,...,...
68,69,595 Dean Street,—,298 (91),27,2022,Topped out in February 2022.[161][162]
69,70 =,Beacon Tower,,297 (91),23,2007,[163][164]
70,70 =,One Northside Piers,,297 (91),29,2008,[165][166]
71,70 =,101 Clark Street,—,295 (90),30,1973,[167][168]


### 9) Which years are represented by at least 5 buildings?

In [146]:
df['Year completed'].value_counts().head(4)

Year completed
2022    11
2020     6
2016     6
2017     4
Name: count, dtype: int64

### 10) How many total floors do all the buildings have, combined?

In [158]:
df['Floors'] = df['Floors'].astype('int')

print(df['Floors'].sum())

2603


### 11) How many of the buildings have their own Wikipedia page?

For this, you'll need to query the row elements again; the information won't have been extracted into your `DataFrame`. (Hint: Whether a building has its own Wikipedia page isn't an explicit piece of data, but something you can infer from the presence of a particular sub-element.)

In [230]:
# for row in soup.select('table')[0]('tr a'):
#     print(row)

# for row in soup.select('table')[0]('tr')[1:]:
#     print (row.select('td')[1]('a'))
#     print('-------')

links = [(row.select('td')[1]('a')) for row in soup.select('table')[0]('tr')[1:] if row.select('td')[1].find('a')]
len(links)

17

### 12) How many have an image?

You could do this by testing for the presence of another element:

In [241]:
for row in soup.select('table')[0]('tr')[1:2]:
    print(row.select('td')[2])

<td><a class="image" href="/wiki/File:The_Brooklyn_Tower_010.jpg" title="City Point"><img alt="A view of The Brooklyn Tower looking north from Bond Street" data-file-height="2048" data-file-width="1536" decoding="async" height="107" src="//upload.wikimedia.org/wikipedia/commons/thumb/3/36/The_Brooklyn_Tower_010.jpg/80px-The_Brooklyn_Tower_010.jpg" srcset="//upload.wikimedia.org/wikipedia/commons/thumb/3/36/The_Brooklyn_Tower_010.jpg/120px-The_Brooklyn_Tower_010.jpg 1.5x, //upload.wikimedia.org/wikipedia/commons/thumb/3/36/The_Brooklyn_Tower_010.jpg/160px-The_Brooklyn_Tower_010.jpg 2x" width="80"/></a>
</td>


In [249]:
# for row in soup.select('table')[0]('tr')[1:]:
#     print(row.select('td')[2])


len ([(row.select('.image')) for row in soup.select('table')[0]('tr')[1:] if row.select('.image')])

56

Or through information that's already in your `DataFrame`:

In [263]:
df['Image'].value_counts()

Image
     56
—    17
Name: count, dtype: int64

### Bonus challenge

If we tried to run the same code on https://en.wikipedia.org/wiki/List_of_tallest_buildings_in_New_York_City instead, the results wouldn't be quite right. Try it. Then, examining the HTML of that page, try to figure out why.

If you want an extra-extra challenge, try writing code that would parse that table correctly.

In [272]:
url2 = requests.get('https://en.wikipedia.org/wiki/List_of_tallest_buildings_in_New_York_City')
html_2 = url2.text
soup2 = BeautifulSoup(html_2)

headers2 = [header.text.strip() for header in soup2.select('table tr')[0]('th')]
buildings2 = [[cell.text.strip() for cell in row.select('td')] for row in soup2.select('table')[0]('tr')[1:]]
len(buildings)

106

In [275]:
df2 = pd.DataFrame(buildings2, columns=headers2)
df2

,Rank,Name,Image,Heightft (m),Floors[H],Year,Address,Coordinates,Notes
0,1,One World Trade Center,,"1,776 (541)",94[A],2014,285 Fulton Street,40°42′47″N 74°00′49″W﻿ / ﻿40.713°N 74.0135°W﻿ ...,Also known as the Freedom Tower. Tallest build...
1,2,Central Park Tower,,"1,550 (472)",99,2021,225 West 57th Street,40°45′57″N 73°58′51″W﻿ / ﻿40.7659°N 73.98089°W...,"Also known as Nordstrom Tower. At 1,550 feet, ..."
2,3,111 West 57th Street,,"1,428 (435)",85,2022,111 West 57th Street,40°45′52″N 73°58′40″W﻿ / ﻿40.76455°N 73.97765°...,Also known as Steinway Tower. It is the world'...
3,4,One Vanderbilt,,"1,401 (427)",73,2020,1 Vanderbilt Avenue,40°45′11″N 73°58′43″W﻿ / ﻿40.7530°N 73.9785°W﻿...,Second tallest office building in NYC.[38] Tal...
4,5,432 Park Avenue,,"1,397 (426)",85,2015,432 Park Avenue,40°45′41″N 73°58′19″W﻿ / ﻿40.761389°N 73.97180...,"Fifth tallest building overall in NYC, third t..."
...,...,...,...,...,...,...,...,...,...
101,102,1 Wall Street,,654 (199),50,1932,1 Wall Street,40°42′26″N 74°00′42″W﻿ / ﻿40.707222°N 74.01166...,It was formerly called Bank of New York Buildi...
102,103 =,599 Lexington Avenue,,653 (199),51,1986,599 Lexington Avenue,40°45′28″N 73°58′15″W﻿ / ﻿40.7578°N 73.9707°W﻿...,[240][241]
103,103 =,Silver Towers I,,653 (199),58,2009,620 West 42nd Street,40°45′39″N 73°59′57″W﻿ / ﻿40.760722°N 73.99919...,Also known as River Place.[242][243]
104,103 =,Silver Towers II,653 (199),58,2009,620 West 42nd Street,40°45′39″N 73°59′57″W﻿ / ﻿40.760722°N 73.99919...,Also known as River Place.[244][245],None


---

---

---